In [1]:
import numpy as np
from pathlib import Path
from astropy.io import fits
from sklearn.cluster import KMeans

fits_dir = Path("/home/yd388/rds/hpc-work/DIS-Project-Lensed-Galaxy/data/raws")
fits_files = sorted(fits_dir.glob("*.fits"))

per_galaxy_p999 = []
skipped = []
for f in fits_files:
    try:
        with fits.open(f) as hdul:
            img = hdul[0].data
            if img is None:
                skipped.append((f.name, "no data in HDU 0"))
                continue
            img = img.astype(np.float32)
        per_galaxy_p999.append(np.percentile(img, 99.9))
    except (OSError, ValueError) as e:
        skipped.append((f.name, f"{type(e).__name__}: {str(e)[:60]}"))

per_galaxy_p999 = np.array(per_galaxy_p999)
print(f"Loaded {len(per_galaxy_p999)} / {len(fits_files)} files")
print(f"Skipped {len(skipped)}:")
for name, reason in skipped[:10]:
    print(f"  {name}: {reason}")

# k-means as before
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(per_galaxy_p999.reshape(-1, 1))
centers = np.sort(km.cluster_centers_.ravel())
A = float(centers.mean())
print(f"\nCluster centers: {centers[0]:.3f}, {centers[1]:.3f}")
print(f"==> A ≈ {A:.2f} ADU")

Loaded 6643 / 12048 files
Skipped 5405:
  Courteau97_UGC1037_g.fits: OSError: Empty or corrupt FITS file
  Courteau97_UGC1037_r.fits: OSError: Empty or corrupt FITS file
  Courteau97_UGC1037_z.fits: OSError: Empty or corrupt FITS file
  Courteau97_UGC10779_g.fits: OSError: Empty or corrupt FITS file
  Courteau97_UGC10779_r.fits: OSError: Empty or corrupt FITS file
  Courteau97_UGC10779_z.fits: OSError: Empty or corrupt FITS file
  Courteau97_UGC12052_g.fits: OSError: Empty or corrupt FITS file
  Courteau97_UGC12052_r.fits: OSError: Empty or corrupt FITS file
  Courteau97_UGC12052_z.fits: OSError: Empty or corrupt FITS file
  Courteau97_UGC12173_g.fits: OSError: Empty or corrupt FITS file

Cluster centers: 1.174, 9.357
==> A ≈ 5.27 ADU
